# 03 · Harnesses, traces and evals

Once a request reaches vLLM it gets GPU time, whether it deserves it or not. So everything that should stop a request happens **before** that, in the gateway: these checks are the **harnesses**. Every request is also a **trace** that follows it from the agent through the gateway and NGINX into vLLM, so when something fails you can see where and why. And an **eval set** checks the whole system's answers automatically, so any change can be measured.

All numbers are read from `results/03_harnesses_and_traces/`.

## The harnesses

All in `gateway/app/harness.py`, configured in `docker-compose.yml`, and switchable off as a group (`HARNESSES_ENABLED=false`) to measure what they do.

| Check | Rejects with | Default limit |
|---|---|---|
| Valid request (JSON, messages present, sane max_tokens) | 400 `invalid_request` | |
| Prompt injection (phrases that try to override the system prompt, in user messages only) | 400 `prompt_injection` | 6 pattern families |
| Output budget (`max_tokens` too big; missing → filled in) | 400 `max_tokens_too_large` | 1,024 tokens |
| Prompt budget (counted exactly with Qwen's tokenizer) | 413 `prompt_too_long` | 6,000 tokens, and prompt + output ≤ 8,192 |
| Per user rate limit (requests and tokens per minute, `x-user-id`) | 429 `rate_limited` + `Retry-After` | 60 requests, 200k tokens |
| Load shedding (requests in flight) | 503 `overloaded` | 256 |
| Time limit (stream cut, upstream cancelled) | 504, or a timeout event mid stream | 120 s, `x-timeout-s` to lower |
| Client disconnect (upstream cancelled, vLLM stops) | | |

Counting prompt tokens exactly matters: Qwen's chat template adds 5 tokens per message, 3 for the assistant turn and a 21 token default system prompt when none is given. With those rules the gateway's count matched vLLM's exactly on every request checked.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = ROOT / "results" / "03_harnesses_and_traces"
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs, summarize

runs = load_runs("03_harnesses_and_traces")
evals = {r["config"]["label"]: r for r in runs if r["name"].startswith("evals_")}
drills = next((r for r in runs if r["name"] == "failure_drills"), None)
agent = next((r for r in runs if r["name"] == "agent_run"), None)
traces = {p.stem: json.loads(p.read_text()) for p in sorted((RESULTS / "traces").glob("*.json"))} if (RESULTS / "traces").exists() else {}
print(f"evals: {sorted(evals)} | drills: {bool(drills)} | agent: {bool(agent)} | traces: {len(traces)}")
if runs:
    env = runs[-1]["environment"]
    gpu = env["gpus"][0] if env["gpus"] else {"name": "no GPU", "memory_mib": 0}
    print(f"hardware: {gpu['name']} | model: {runs[-1]['config'].get('model')} | vLLM config: {runs[-1]['config'].get('vllm_config')}")

## Evals: does the system give the right answers?

22 cases in `evals/eval_set.json`, scored automatically by `evals/run_evals.py` (text matching, regular expressions, JSON parsing, refusal detection; no human or LLM judge).

In [ ]:
if "harnesses_on" not in evals:
    print("No eval runs yet: run scripts/run_phase3.sh on the GPU box, then copy results/ back.")
else:
    run = evals["harnesses_on"]["metrics"]
    print(f"{run['summary']['passed']}/{run['summary']['total']} passed\n")
    for cat, c in run["summary"]["by_category"].items():
        print(f"  {cat:<14}{c['passed']}/{c['total']}")
    print()
    for case in run["cases"]:
        print(f"{'PASS' if case['passed'] else 'FAIL'}  {case['category']:<13} {case['id']:<28} {case['why']}")

## Harnesses on vs off

The same 22 requests with every harness switched off. For the harness cases, this shows what would have happened without them: what the model did, and the GPU time it cost.

In [ ]:
if len(evals) == 2:
    on, off = evals["harnesses_on"]["metrics"], evals["harnesses_off"]["metrics"]
    off_cases = {c["id"]: c for c in off["cases"]}
    print(f"{'harness case':<28}{'on: status':>12}{'off: status':>13}{'off: GPU time':>15}{'off: tokens out':>17}  secret leaked (on / off)")
    for case in on["cases"]:
        if case["category"] != "harness":
            continue
        o = off_cases[case["id"]]
        print(f"{case['id']:<28}{case['result']['status']:>12}{o['result']['status']:>13}{o['gpu_s'] * 1000:>12.0f} ms"
              f"{o['engine']['generation_tokens']:>17.0f}  {case['leaked_secret']} / {o['leaked_secret']}")
    blocked_ids = [c["id"] for c in on["cases"] if c["expect"]["type"] == "blocked"]
    saved = sum(off_cases[i]["gpu_s"] for i in blocked_ids)
    print(f"\nGPU time the blocked requests would have used: {saved:.2f} s for {len(blocked_ids)} requests")
    print(f"Secrets leaked: {on['summary']['secrets_leaked']} with harnesses, {off['summary']['secrets_leaked']} without")
else:
    print("Needs both the harnesses_on and harnesses_off runs.")

## Failure drills

`scripts/failure_drills.py` triggers each kind of failure on purpose and checks that it is caught, cheap, and visible in the metrics.

In [ ]:
if drills:
    m = drills["metrics"]
    print(f"{m['passed']}/{m['total']} drills passed\n")
    for d in m["drills"]:
        print(f"{'PASS' if d['passed'] else 'FAIL'}  {d['drill']:<12} expected: {d['expected']}")
        detail = {k: (round(v, 1) if isinstance(v, float) else v) for k, v in d.items()
                  if k not in ("drill", "expected", "passed", "body", "metrics_change", "trace_id")}
        print(f"      observed: {detail}")
        print(f"      metrics:  {', '.join(f'{k} +{v:g}' for k, v in d['metrics_change'].items()) or 'none'}")
else:
    print("No drill run yet.")

## Traces

Each saved trace is the same data Jaeger shows: one bar per span, nested by parent, colored by service.

In [ ]:
COLORS = {"agent": "#6a51a3", "gateway": "#2171b5", "nginx": "#41ab5d", "vllm": "#d94801"}

def spans_of(trace):
    out = []
    for r in trace["result"]["resourceSpans"]:
        svc = next(a["value"]["stringValue"] for a in r["resource"]["attributes"] if a["key"] == "service.name")
        for ss in r["scopeSpans"]:
            for s in ss["spans"]:
                out.append({"service": svc, "name": s["name"], "id": s["spanId"], "parent": s.get("parentSpanId", ""),
                            "start": int(s["startTimeUnixNano"]), "end": int(s["endTimeUnixNano"]),
                            "attrs": {a["key"]: next(iter(a["value"].values())) for a in s.get("attributes", [])}})
    return out

def ordered(spans):
    ids = {s["id"] for s in spans}
    def walk(parent, depth):
        for s in sorted((s for s in spans if s["parent"] == parent), key=lambda s: s["start"]):
            yield s, depth
            yield from walk(s["id"], depth + 1)
    for root in sorted((s for s in spans if s["parent"] not in ids), key=lambda s: s["start"]):
        yield root, 0
        yield from walk(root["id"], 1)

def waterfall(name, title):
    if name not in traces:
        print(f"trace {name} not found")
        return
    rows = list(ordered(spans_of(traces[name])))
    t0 = min(s["start"] for s, _ in rows)
    fig, ax = plt.subplots(figsize=(11, 0.32 * len(rows) + 1))
    for i, (s, depth) in enumerate(rows):
        start, dur = (s["start"] - t0) / 1e6, (s["end"] - s["start"]) / 1e6
        ax.barh(i, max(dur, 0.2), left=start, color=COLORS.get(s["service"], "#999"))
        ax.text(start + max(dur, 0.2), i, f"  {dur:.1f} ms", va="center", fontsize=8)
    ax.set_yticks(range(len(rows)), [("  " * d) + f"{s['service']}: {s['name']}" for s, d in rows], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("ms since the trace started")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

agent_trace = next((k for k in traces if k.startswith("agent_")), None)
if agent_trace:
    waterfall(agent_trace, "One agent run: 3 LLM calls, each through gateway → NGINX → vLLM")

In [ ]:
for name, title in [("eval_harnesses_on_correct_capital", "One normal request"),
                    ("drill_injection", "A request stopped by the prompt injection harness"),
                    ("drill_timeout", "A request cut off by the 2 s time limit")]:
    waterfall(name, title)

### What vLLM records on its span

vLLM's `llm_request` span carries its own timing for the request, joined to the same trace as the gateway and NGINX spans.

In [ ]:
vllm_spans = [s for t in traces.values() for s in spans_of(t) if s["service"] == "vllm"]
if vllm_spans:
    s = vllm_spans[0]
    for k, v in sorted(s["attrs"].items()):
        if k.startswith("gen_ai"):
            print(f"{k:<45}{v}")
else:
    print("No vLLM spans in the saved traces.")

### What the harness costs

The `harness` span times the checks on every request that passed through them.

In [ ]:
harness = [(s["end"] - s["start"]) / 1e6 for t in traces.values() for s in spans_of(t) if s["name"] == "harness"]
if harness:
    h = summarize(harness)
    print(f"harness checks: {h['count']} spans, p50 {h['p50']:.2f} ms, p95 {h['p95']:.2f} ms, max {h['max']:.2f} ms")

## Grafana and Jaeger

*(added after the GPU session)*

## What happened

*(written after the GPU session, from the numbers above)*

## Phase 3 exit criteria

- [x] Harnesses in the gateway: validation, prompt injection, token budgets, rate limits, load shedding, time limits, client disconnects
- [x] Traces across agent → gateway → NGINX → vLLM in Jaeger, saved to `results/`
- [x] Eval set with automatic scoring, runnable after every change (`make eval`)
- [x] Failure drills for every failure type, each with a trace and a dashboard signal
- [ ] Phase 3 session on the RTX 5090: evals (harnesses on and off), drills, traced agent run
- [ ] Every failure type explained: caught where, how fast, what it saved